In [1]:
import geopandas as gpd
import ee
import geemap
# ee.Authenticate()
# ee.Initialize()
import ipyleaflet
import shapely
import json
import rasterio
import matplotlib.pyplot as plt
import geojson

In [2]:
#create empty map
Map = geemap.Map()

gdf=gpd.read_file('/Users/isabellopez/Desktop/SURGE/notebooks/exploration/data/DISTRICT2011.shp')

jaipur= gdf[gdf['NAME']=='Jodhpur']
def shapely_to_geojson(geometry):
    return geojson.dumps(geometry)
jaipur_json=json.loads(shapely_to_geojson(jaipur['geometry'].values[0]))
roi= ee.Geometry.Polygon(jaipur_json['coordinates'])


# center map on selected country
Map.centerObject(roi, 7)

Map

Map(center=[26.74643615109251, 72.78694440076592], controls=(WidgetControl(options=['position', 'transparent_b…

In [3]:
landcover= ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterDate("2015-01-1","2023-05-19").filterBounds(roi).select('label')

In [4]:

crop_class = 4
# Create a binary layer where 1 represents crop and 0 represents all other landcover types
landcover_crop = landcover.map(lambda image: image.eq(crop_class))

# Define the visualization parameters for the crop layer
cropVis = { "min":0, "max":1, 'palette': ['black', '#E49635']}

# Add the crop layer to the map
Map.addLayer(landcover_crop.mosaic().clip(roi), cropVis, "Crop Landcover")

legend_dict = {
    'Crops': '#E49635',
}

Map.add_legend(legend_title="Land Cover of Jaipur", legend_dict=legend_dict)


In [5]:
# task=ee.batch.Export.image.toDrive(landcover.mosaic(), fileNamePrefix='landcover_jaipur_filter',folder= 'ee', scale=1000, region=jaipur_json['coordinates'])
# task.start()

In [5]:
skysat= ee.ImageCollection("SKYSAT/GEN-A/PUBLIC/ORTHO/RGB").filterDate("2023-03-1","2023-06-30").filterBounds(roi).select(['R','G', 'B'])

# Define the visualization parameters for the crop layer
skysatVis = { "min":11, "max":190}

# Add the crop layer to the map
Map.addLayer(skysat.mosaic().clip(roi), skysatVis, "SkySat")